# Step 4: After 평가 (Fine-tuned 모델)

KoDF로 Fine-tuning된 모델의 성능을 평가하고 **Model Registry에 등록**합니다.

## 실습 목표
- Fine-tuned 모델 평가
- **SageMaker Model Registry 등록**
- 모델 버전 관리

## 예상 결과: ~90%+ 정확도

In [ ]:
import json
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import boto3
import tarfile
from pathlib import Path

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path(os.getcwd()).parent

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Config loaded from: {config_path}")

In [ ]:
# S3에서 Fine-tuned 모델 다운로드
import sagemaker
s3 = boto3.client('s3')

model_data = config['model_data']
print(f"모델 경로: {model_data}")

# model.tar.gz 다운로드 및 압축 해제
!aws s3 cp {model_data} ./model.tar.gz
!tar -xzf model.tar.gz -C ./
print("모델 다운로드 완료")

In [ ]:
# Fine-tuned 모델 로드
class DeepfakeDetector(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=2, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    def forward(self, x):
        return self.backbone(x)

model_after = DeepfakeDetector(pretrained=False)
model_after.load_state_dict(torch.load('./model.pth', map_location=device))
model_after = model_after.to(device)
model_after.eval()
print("Fine-tuned 모델 로드 완료")

In [ ]:
# 테스트 데이터 로드
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# config에서 경로 가져오기
test_data_path = config.get('local_test_path', str(PROJECT_ROOT / '1_data_preparation' / 'data' / 'test'))
print(f"테스트 데이터 경로: {test_data_path}")

test_dataset = datasets.ImageFolder(test_data_path, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# 평가 함수 (2_before_evaluation과 동일)
def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(data_loader):
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='binary', pos_label=0),
        'recall': recall_score(all_labels, all_preds, average='binary', pos_label=0),
        'f1_score': f1_score(all_labels, all_preds, average='binary', pos_label=0),
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist()
    }

after_results = evaluate_model(model_after, test_loader, device)

In [ ]:
print("\n" + "="*50)
print("📊 AFTER 모델 평가 결과 (KoDF Fine-tuned)")
print("="*50)
print(f"  Accuracy:  {after_results['accuracy']*100:.1f}%")
print(f"  Precision: {after_results['precision']*100:.1f}%")
print(f"  Recall:    {after_results['recall']*100:.1f}%")
print(f"  F1 Score:  {after_results['f1_score']*100:.1f}%")
print("="*50)
print("✅ 한국인 특화 Fine-tuning 효과!")
print("="*50)

In [ ]:
# 결과 저장
with open('after_results.json', 'w') as f:
    json.dump({'model_type': 'after', **after_results}, f, indent=2)
print("결과 저장 완료: after_results.json")

# config 업데이트
config['after_accuracy'] = after_results['accuracy']
with open('../config.json', 'w') as f:
    json.dump(config, f, indent=2)

## 4.1 Model Registry 등록

성능이 기준(85%)을 넘으면 Model Registry에 등록합니다.

In [ ]:
from sagemaker.pytorch import PyTorchModel

MODEL_PACKAGE_GROUP = "deepfake-detection-kodf"
ACCURACY_THRESHOLD = 0.85

# Model Package Group 생성 (처음 한 번만)
sm_client = boto3.client('sagemaker')
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="한국인 딥페이크 탐지 모델"
    )
    print(f"Model Package Group 생성: {MODEL_PACKAGE_GROUP}")
except sm_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Model Package Group 이미 존재: {MODEL_PACKAGE_GROUP}")
    else:
        raise e

In [ ]:
# 조건부 Model Registry 등록
if after_results['accuracy'] >= ACCURACY_THRESHOLD:
    print(f"✅ 정확도 {after_results['accuracy']*100:.1f}% >= {ACCURACY_THRESHOLD*100}% 기준 충족!")
    print("Model Registry에 등록합니다...")
    
    # inference.py 경로
    inference_dir = str(PROJECT_ROOT / '6_demo')
    
    # PyTorch 모델 정의
    pytorch_model = PyTorchModel(
        model_data=config['model_data'],
        role=config['role'],
        framework_version='2.0.0',
        py_version='py310',
        entry_point='inference.py',
        source_dir=inference_dir
    )
    
    # Model Registry 등록
    model_package = pytorch_model.register(
        model_package_group_name=MODEL_PACKAGE_GROUP,
        inference_instances=['ml.g4dn.xlarge', 'ml.m5.large'],
        transform_instances=['ml.m5.large'],
        content_types=['application/x-image', 'application/json'],
        response_types=['application/json'],
        approval_status='PendingManualApproval',
        description=f"KoDF Fine-tuned (Accuracy: {after_results['accuracy']*100:.1f}%)"
    )
    
    print(f"✅ Model Registry 등록 완료!")
    print(f"   Model Package ARN: {model_package.model_package_arn}")
    config['model_package_arn'] = model_package.model_package_arn
    
else:
    print(f"❌ 정확도 {after_results['accuracy']*100:.1f}% < {ACCURACY_THRESHOLD*100}% 기준 미달")
    print("Model Registry 등록 건너뜀")

# config 저장
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)